# 02 · Transformação: camada Silver

**Objetivo da etapa:** converter os dados brutos da Bronze em tabelas **limpas, tipadas e
conciliadas**. Todas as regras de negócio desta etapa estão listadas abaixo, porque afetam
diretamente a interpretação dos resultados.

| Regra | Descrição | Justificativa |
|---|---|---|
| **R01 · Tipagem** | Datas e horas viram `timestamp`, a data prometida vira `date`, valores viram `double` e contagens viram `int`. | A Bronze guarda tudo como texto. |
| **R02 · Chaves de CEP** | O prefixo de CEP é tratado como texto de 5 dígitos, completado com zeros à esquerda. | Preserva prefixos como `01001`. |
| **R03 · Padronização de texto** | UF em maiúsculas; cidade e categoria em minúsculas, sem espaços nas pontas. | Evita grupos duplicados por grafia. |
| **R04 · Nomes em português** | Colunas renomeadas para o português, e erros de digitação da origem corrigidos (`lenght` → `length`). | Legibilidade do modelo e do relatório. |
| **R05 · Geolocalização** | São descartadas as coordenadas fora do retângulo do território brasileiro. Cada prefixo de CEP recebe a **mediana** das latitudes e longitudes. A cidade da tabela de geolocalização é descartada: tem grafia inconsistente e a cidade já vem nas tabelas de clientes e vendedores. | A base tem 1 milhão de linhas com duplicatas e pontos no exterior; a mediana é robusta a valores extremos. |
| **R06 · Uma avaliação por pedido** | Quando um pedido tem mais de uma avaliação, fica a de resposta mais recente. | Evita duplicar pedidos na tabela fato. |
| **R07 · Produtos** | Peso ou dimensão igual a 0 vira nulo, por ser fisicamente impossível. Categoria ausente vira `sem_categoria`. As 2 categorias sem tradução recebem tradução manual. | Acurácia e completude. |
| **R08 · Consistência temporal** | Os pedidos com datas contraditórias são **sinalizados** (colunas `dq_*`), e não apagados. As análises de prazo excluem os pedidos sinalizados. | Não existe fonte para corrigir a data verdadeira; a sinalização é auditável. |

A camada Silver mantém o **grão original** de cada tabela, sem agregações. O modelo
dimensional é montado na etapa 03.

### Configuração do ambiente
Esta célula prepara o ambiente de execução. No Google Colab, ela:
1. clona o repositório do projeto, que contém o módulo `scripts/common.py`;
2. instala o PySpark e o Delta Lake;
3. monta o Google Drive, onde o lakehouse fica persistido.

Na primeira execução, o Colab pede autorização de acesso ao Drive.

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/FilippeDumont/mvp-olist-atrasos-entrega.git"
REPO_DIR = "/content/repo"

if "google.colab" in sys.modules:
    if not os.path.isdir(f"{REPO_DIR}/.git"):
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
    # O Spark precisa de Java: instala um JDK apenas se a máquina não tiver um
    if subprocess.run(["bash", "-c", "command -v java"], capture_output=True).returncode != 0:
        subprocess.run(["bash", "-c", "apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pyspark==3.5.3", "delta-spark==3.2.1", "kagglehub"], check=True)
    sys.path.insert(0, f"{REPO_DIR}/scripts")
else:  # execução local de teste
    sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "scripts")))

import common as C

C.mount_drive()
spark = C.get_spark()
print("Spark", spark.version, "| Delta Lake habilitado")

In [ ]:
from pyspark.sql import functions as F, Window as W

C.banner("02 · Transformação (Silver)")
b = {t: C.read_table("bronze", t) for t in
     ["orders", "customers", "order_items", "order_payments", "order_reviews",
      "products", "sellers", "geolocation", "category_translation"]}

ts = lambda c: F.to_timestamp(F.col(c), "yyyy-MM-dd HH:mm:ss")
cep = lambda c: F.lpad(F.trim(F.col(c)), 5, "0")
txt = lambda c: F.lower(F.trim(F.col(c)))
uf = lambda c: F.upper(F.trim(F.col(c)))

## Pedidos (R01, R08)

In [ ]:
pedidos = (b["orders"].select(
        F.col("order_id").alias("pedido_id"),
        F.col("customer_id").alias("cliente_id"),
        txt("order_status").alias("status"),
        ts("order_purchase_timestamp").alias("dt_compra"),
        ts("order_approved_at").alias("dt_aprovacao"),
        ts("order_delivered_carrier_date").alias("dt_postagem"),
        ts("order_delivered_customer_date").alias("dt_entrega"),
        F.to_date(ts("order_estimated_delivery_date")).alias("dt_entrega_prometida"))
    # R08: sinalização de inconsistências temporais
    .withColumn("dq_postagem_antes_compra", F.coalesce(F.col("dt_postagem") < F.col("dt_compra"), F.lit(False)))
    .withColumn("dq_entrega_antes_postagem", F.coalesce(F.col("dt_entrega") < F.col("dt_postagem"), F.lit(False)))
    .withColumn("dq_entregue_sem_data", (F.col("status") == "delivered") & F.col("dt_entrega").isNull())
    .withColumn("dq_data_entrega_sem_status", (F.col("status") != "delivered") & F.col("dt_entrega").isNotNull())
    .withColumn("dq_inconsistente", F.col("dq_postagem_antes_compra") | F.col("dq_entrega_antes_postagem")
                | F.col("dq_entregue_sem_data") | F.col("dq_data_entrega_sem_status")))

## Clientes e vendedores (R02, R03)

In [ ]:
clientes = b["customers"].select(
    F.col("customer_id").alias("cliente_id"),
    F.col("customer_unique_id").alias("cliente_unico_id"),
    cep("customer_zip_code_prefix").alias("cep_prefixo"),
    txt("customer_city").alias("cidade"),
    uf("customer_state").alias("uf"))

vendedores = b["sellers"].select(
    F.col("seller_id").alias("vendedor_id"),
    cep("seller_zip_code_prefix").alias("cep_prefixo"),
    txt("seller_city").alias("cidade"),
    uf("seller_state").alias("uf"))

## Itens, pagamentos e avaliações (R01, R06)

In [ ]:
itens = b["order_items"].select(
    F.col("order_id").alias("pedido_id"),
    F.col("order_item_id").cast("int").alias("item_seq"),
    F.col("product_id").alias("produto_id"),
    F.col("seller_id").alias("vendedor_id"),
    ts("shipping_limit_date").alias("dt_limite_postagem"),
    F.col("price").cast("double").alias("preco"),
    F.col("freight_value").cast("double").alias("frete"))

pagamentos = b["order_payments"].select(
    F.col("order_id").alias("pedido_id"),
    F.col("payment_sequential").cast("int").alias("pagamento_seq"),
    txt("payment_type").alias("tipo_pagamento"),
    F.col("payment_installments").cast("int").alias("parcelas"),
    F.col("payment_value").cast("double").alias("valor"))

av = b["order_reviews"].select(
    F.col("review_id").alias("avaliacao_id"),
    F.col("order_id").alias("pedido_id"),
    F.col("review_score").cast("int").alias("nota"),
    F.col("review_comment_title").alias("titulo"),
    F.col("review_comment_message").alias("comentario"),
    ts("review_creation_date").alias("dt_criacao"),
    ts("review_answer_timestamp").alias("dt_resposta"))
# R06: 1 avaliação por pedido (a respondida mais recentemente)
w = W.partitionBy("pedido_id").orderBy(F.col("dt_resposta").desc(), F.col("dt_criacao").desc(), F.col("avaliacao_id"))
avaliacoes = (av.withColumn("_rk", F.row_number().over(w)).filter("_rk = 1").drop("_rk")
              .withColumn("tem_comentario", F.length(F.trim(F.coalesce("comentario", F.lit("")))) > 0))
print(f"Avaliações: {av.count():,} na origem -> {avaliacoes.count():,} após R06 (1 por pedido)")

## Produtos (R04, R07)

In [ ]:
traducao_manual = spark.createDataFrame(
    [("pc_gamer", "pc_gamer"),
     ("portateis_cozinha_e_preparadores_de_alimentos", "portable_kitchen_and_food_preparers")],
    "product_category_name string, product_category_name_english string")
traducao = (b["category_translation"].select("product_category_name", "product_category_name_english")
            .unionByName(traducao_manual).dropDuplicates(["product_category_name"]))

zero_nulo = lambda c: F.when(F.col(c).cast("double") > 0, F.col(c).cast("double"))
produtos = (b["products"].join(traducao, "product_category_name", "left").select(
    F.col("product_id").alias("produto_id"),
    F.coalesce(txt("product_category_name"), F.lit("sem_categoria")).alias("categoria"),
    F.coalesce(txt("product_category_name_english"), F.lit("no_category")).alias("categoria_en"),
    F.col("product_name_lenght").cast("int").alias("qtd_caracteres_nome"),
    F.col("product_description_lenght").cast("int").alias("qtd_caracteres_descricao"),
    F.col("product_photos_qty").cast("int").alias("qtd_fotos"),
    zero_nulo("product_weight_g").alias("peso_g"),
    zero_nulo("product_length_cm").alias("comprimento_cm"),
    zero_nulo("product_height_cm").alias("altura_cm"),
    zero_nulo("product_width_cm").alias("largura_cm")))

## Geolocalização por prefixo de CEP (R05)
O retângulo do território brasileiro vai de 33,75°S a 5,27°N de latitude e de 73,99°W a 34,79°W
de longitude. Os limites usados abaixo têm uma pequena margem.

In [ ]:
geo_raw = b["geolocation"].select(
    cep("geolocation_zip_code_prefix").alias("cep_prefixo"),
    F.col("geolocation_lat").cast("double").alias("lat"),
    F.col("geolocation_lng").cast("double").alias("lng"),
    uf("geolocation_state").alias("uf"))
dentro_br = F.col("lat").between(-33.80, 5.30) & F.col("lng").between(-74.00, -34.70)
n_total, n_fora = geo_raw.count(), geo_raw.filter(~dentro_br).count()
geo = (geo_raw.filter(dentro_br).groupBy("cep_prefixo")
       .agg(F.expr("percentile(lat, 0.5)").alias("lat"),
            F.expr("percentile(lng, 0.5)").alias("lng"),
            F.count("*").alias("qtd_registros_origem"),
            F.mode("uf").alias("uf")))
print(f"Geolocalização: {n_total:,} linhas, {n_fora:,} fora do Brasil descartadas -> {geo.count():,} prefixos de CEP")

## Gravação na camada Silver

In [ ]:
silver = {"pedidos": pedidos, "clientes": clientes, "vendedores": vendedores, "itens_pedido": itens,
          "pagamentos": pagamentos, "avaliacoes": avaliacoes, "produtos": produtos,
          "geolocalizacao_cep": geo}
for nome, df in silver.items():
    with C.Stage("02_transformacao", f"silver.{nome}") as st:
        st.rows = C.write_table(df, "silver", nome)

## Resumo das sinalizações de consistência (R08)

In [ ]:
C.read_table("silver", "pedidos").select(
    *[F.sum(F.col(c).cast("int")).alias(c) for c in
      ["dq_postagem_antes_compra", "dq_entrega_antes_postagem", "dq_entregue_sem_data",
       "dq_data_entrega_sem_status", "dq_inconsistente"]]).show()